# nano Beatrix v1 — four bodies through Beatrix 2s's diet

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_2s.ipynb)

Four small byte-level crafts train from birth on the data Beatrix 2s trained on: 16.1 billion bytes in thirteen phases, with the AlephLLM library's own trainer and every curve it records. Each craft is one body of the nano bed, started from exactly the weights its strongest bed run started from.

| craft | the body | its bed run (the strongest seed) |
|---|---|---|
| `nano-pollux-dual-canon-v1` | Pollux's layout (splat hubs at blocks 0, 2 and 4; rotary softmax at 1, 3 and 5) with dual hubs and Canon layers | `altdc-both`, seed 1 |
| `nano-pollux-canon-v1` | Pollux (rotary hubs and rotary softmax alternating) with Canon layers | `polluxc-both`, seed 2 |
| `nano-pollux-dual-v1` | Pollux's layout with dual hubs | `altd-both`, seed 1 |
| `nano-castor-v1` | Castor: rotary softmax in every block (the baseline) | `castor-both`, seed 3 |

Every craft is 256 wide with six blocks, eight heads and a bank of seven experts with the byte table added: about 9.2 million parameters and a 4,096-byte window. A dual hub keeps two address masses in one block, one rotary (sharp nearby) and one plain (flat with distance). A Canon layer mixes each byte's vector with the three before it, channel by channel, before the mixer and before the bank.

**The diet** (Beatrix 2s's, in order):

| phase | data | bytes |
|---|---|---|
| `warmup_wikitext` | wikitext-103 | 0.3B |
| `fineweb_main` | fineweb-edu | 5.0B |
| `curriculum_s0` to `curriculum_s8` | the nine curriculum mixes (stories, simple English, concepts, rules, arithmetic, cause and effect, trying and failing, definitions and register; recall rows throughout) | 8.8B |
| `anneal_nochat` | the anneal mix without chat | 1.0B |
| `anneal_mix` | the anneal mix | 1.0B |

**The recipe** is Beatrix 2s's: Muon at 0.02 on the trunk's matrices and plain Adam at 0.0003 on the rest, 200 warmup steps then a flat rate, gradient clip 1.0, 262,144 bytes a step: 61,422 steps a craft.

**The curves** (TensorBoard, one run per craft):

| curves | what they read | how often |
|---|---|---|
| `train/*` | loss, bits a byte, gradient norm, learning-rate scale, tokens, tokens a second | every 50 steps |
| the health census | each block's address health, consumed rank, hidden rank and bank dispatch | every 500 steps |
| the full eval | bits a byte on the fineweb-edu holdout (Beatrix 2s's gauge), the toggle ledger (the bank, the hubs, the head address off), canaries, the document-boundary gauge, bits by position band in the window (0-64 to 2,048-4,096) | every 2,000 steps and at every phase boundary |
| `bed/*` | the nano bed's gauges: the keyed record by frame size from 32 to 1,024 bytes (`bed/scaling`), the copy and the nested and redefined-key questions (`bed/frame`), a planted record at distances out to 3,584 bytes and the bits by position (`bed/window`), canary rows, bits by byte class and with the bank off (`bed/text`) | every 4,000 steps and at every phase boundary |

The 2s diet holds none of the bed's framed rows, so the frame reads show what the diet alone teaches about finding a record in context.

**The order:** all four crafts go through a phase before any goes on, so the four curves stay comparable whenever the run stops.

**Length:** C2 measures each craft's step on the card in hand and prints the estimate. The bed measured these bodies at 0.6 to 0.9 million bytes a second on an RTX PRO 6000; at that pace the four crafts' 64.4 billion bytes take about a day. Curriculum stage 8 reads a slow corpus (definitions), so it may run at the data's pace.

**Shipping and resuming:** each craft's checkpoints (every 2,000 steps), resume state, manifest, TensorBoard files and gauge records go to [AbstractPhil/alephllm-nano-castor-pollux](https://huggingface.co/AbstractPhil/alephllm-nano-castor-pollux) under its name. Rerunning C3 resumes, and so does a new session. A stop (the cell's stop button) saves the running craft's state first.

**Token:** put a Hugging Face token with write access to that repo in this notebook's secrets as `HF_TOKEN` before C1. Without one, everything runs and nothing ships (and nothing resumes in a new session).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.27"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 27), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN                 # the data streams read the Hub authenticated (its rate limits are far higher)

# ============ THE DECISION BLOCK — every setting from the record, with its source beside it ============
OUT = '/content/nano_v1'                              # the session folder: the births, the runs, the bed's gauge folder
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux: each craft under its own name
NAMES = list(V1.CRAFTS)                               # the four crafts, in the round-robin's order (the strongest seed of each body)
SHAPES = [(16, 4), (32, 2), (64, 1)]                  # rows a micro-batch x micro-batches a step, each 262,144 bytes a step; C2 benches
                                                      # them and keeps the 2s shape (16 x 4) unless another is 10% faster and fits
GAUGE_EVERY = V1.GAUGE_EVERY                          # the bed's gauges every 4,000 steps (two full evals) and at every boundary
MAX_HOURS = None                                      # this session's budget in hours (None: until the diet ends or the cell is stopped)
print('decision block: crafts', ', '.join(f"{n} ({V1.CRAFTS[n]['arm']} seed {V1.CRAFTS[n]['seed']})" for n in NAMES),
      '| shapes', SHAPES, '| bed gauges every', GAUGE_EVERY, 'steps | out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the births (each the start of its bed run, checked), the held-out head for the bed's gauges, the 2s step of every
# craft on this card at each shape (the shape is picked here), the length, the push probe. Nothing trains here.
import json, time
from geolip.nano_beatrix.bed_corpus import write_heldout
births = {n: V1.write_birth(n, f'{OUT}/births') for n in NAMES}
held = write_heldout(f'{OUT}/bed')
print('births:', ', '.join(f'{n} <- {V1.CRAFTS[n]["arm"]} seed {V1.CRAFTS[n]["seed"]}' for n in NAMES), '| held-out head', os.path.basename(held))

card_gb = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
BENCH = []
for n in NAMES:
    for mb, ga in SHAPES:
        r = V1.bench(n, births[n], mb, ga)
        BENCH.append(r)
        print(f'  {n:<26} {mb:>2} x {ga}: {r["tok_s"] / 1e3:6.0f}k bytes a second, {r["step_s"]:.3f} s a step, peak {r["peak_gb"]:.1f} GB')
(MB, GA), why = V1.pick_shape(BENCH, card_gb)
print(f'-> every craft trains at {MB} x {GA}: {why}')
steps = 61_422
secs = {r['name']: r['step_s'] * steps for r in BENCH if (r['micro_batch'], r['grad_accum']) == (MB, GA)}
for n in NAMES:
    print(f'  {n:<26} {secs[n] / 3600:5.1f} h of training')
print(f'-> the diet trains about {sum(secs.values()) / 3600:.1f} h on this card, plus about a minute a gauge (about 21 a craft) and the '
      'evals; curriculum stage 8 may run at its data\'s pace')

# the push probe (the ship-complete law): a write to the repo and a read-back BEFORE any training. Tokenless = nothing ships.
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'preflight/push_probe.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'bench': BENCH, 'shape': [MB, GA], 'crafts': {n: V1.CRAFTS[n] for n in NAMES}}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano v1: push probe')
    files = set(api.list_repo_files(REPO))
    PUSH_OK = path in files
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
    if PUSH_OK and 'README.md' not in files:          # the repo's card while the crafts train (the AutoModel card replaces it)
        api.upload_file(path_or_fileobj=V1.card_in_training().encode(), path_in_repo='README.md', repo_id=REPO,
                        commit_message='nano v1: the card while the crafts train')
        print('the card written: README.md')
else:
    print('TOKENLESS: no probe written; C3 runs locally and ships nothing')

In [ ]:
# C3 (T) — THE TRAIN: round-robin by phase (every craft through a phase before any goes on); each craft resumes from its record in the
# repo; the trainer's own lines (the step bar, the health readouts, the evals, the checkpoints) and the bed's gauge lines ([v1]) print as
# they come; a BOUNDARY returns to the driver, which evals, gauges, ships the craft's boundary table and TensorBoard, and moves on.
# Rerun the cell to continue (a new session too). The cell's stop button saves the running craft's state before it stops.
HIST = V1.run(OUT, HF_TOKEN if PUSH_OK else None, NAMES, micro_batch=MB, grad_accum=GA, repo=REPO, gauge_every=GAUGE_EVERY,
              max_hours=MAX_HOURS)
from IPython.display import Markdown, display
display(Markdown(V1.table(HIST)))

In [ ]:
# C4 — THE CURVES: every craft's TensorBoard files from the repo (all sessions; tokenless: this session's local files) and the boundary
# table. Run any time; it trains nothing.
import json
from IPython.display import Markdown, display
logdir = f'{OUT}/runs'
hist = {}
if HF_TOKEN:
    from huggingface_hub import snapshot_download
    hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                            allow_patterns=[f'{n}/runs/*' for n in NAMES] + [f'{n}/v1_boundaries.json' for n in NAMES])
    logdir = hub
    for n in NAMES:
        p = f'{hub}/{n}/v1_boundaries.json'
        hist[n] = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else []
elif os.path.exists(f'{OUT}/v1_boundaries.json'):
    hist = json.load(open(f'{OUT}/v1_boundaries.json', encoding='utf-8'))
display(Markdown(V1.table(hist)))
%load_ext tensorboard
%tensorboard --logdir {logdir}

### After the diet: the AutoModel packages (C5)

When all four crafts have finished, run C1 and then C5 (a fresh session is fine; in the session that trained, C1 asks for a runtime restart first when the package version changed). C5 turns the repo into `transformers` AutoModel packages:

| where | what |
|---|---|
| the repo's top | the model code (transformers looks for remote code at the top of a repo even when the weights come from a subfolder) and the card |
| each craft's folder | `config.json`, `model.safetensors` (bf16, the default), `model.fp32.safetensors` (the fp32 weights the train ended with), a copy of the model code (so a folder downloaded alone loads too), beside the training record |

Nothing is pushed unless every gate passes on exactly the files being pushed: the vendored code is complete, the bf16 file is the fp32 file rounded bit for bit, the vendored code gives the library's logits exactly, and transformers loads each craft in a separate process where the library cannot be imported, by both routes and from both files, with the same logits. After the push the load gate runs again from the repo itself. Then each craft loads with

```python
AutoModelForCausalLM.from_pretrained('AbstractPhil/alephllm-nano-castor-pollux', subfolder='nano-castor-v1', trust_remote_code=True)
```

The TensorBoard curves are already in the repo under each craft's `runs/`.

In [ ]:
# C5 — AFTER THE DIET: the crafts as transformers AutoModel packages in the repo. Each craft's final files come from the repo, every
# gate runs (the code, the config, the weights, exact parity with the library, a transformers load in a separate process), the card is
# written from the boundary tables, one commit pushes it all, and the load gate runs again from the repo. Trains nothing.
%pip install -q "transformers==5.18.0"
from geolip.nano_beatrix import automodel as AM
assert HF_TOKEN, 'the packages are written to the repo: put a token with write access in the secrets as HF_TOKEN, then rerun C1'
COMMIT = AM.release(OUT, REPO, NAMES, HF_TOKEN)
print(f'the AutoModel packages are in the repo: https://huggingface.co/{REPO}/commit/{COMMIT}')

### Changing the train

Edit the decision block in C1, rerun C1 (the install is cached), then C2 and C3. A craft already in the repo resumes; it never restarts.

| to | the edit in C1 |
|---|---|
| fewer crafts this session | `NAMES = ['nano-castor-v1', 'nano-pollux-dual-canon-v1']` (the others wait where they are) |
| a session budget | `MAX_HOURS = 10` |
| the 2s shape whatever the bench says | `SHAPES = [(16, 4)]` |

The session folder is the Colab disk: the repo holds the records.